# Dependencies

In [1]:
!pip install tensorflow opencv-python mediapipe scikit-learn numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.3 MB/s eta 0:00:00m eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 1.2 MB/s eta 0:00:00m eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 4.9 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 5.6 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 6.5 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 6.7 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 5.9 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.5 MB/s eta 0:00:00


In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow.keras.utils import to_categorical

2026-01-01 17:32:49.802047: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-01 17:32:49.859238: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-01 17:32:51.374887: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# Hand Landmarker

In [13]:
from typing import Literal, Protocol, Sequence, runtime_checkable
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


ImageArray = np.ndarray
TimestampMs = int
FeatureVector = np.ndarray

HandednessLabel = Literal["Left", "Right"]

@runtime_checkable
class Category(Protocol):
    index: int
    score: float
    category_name: HandednessLabel


@runtime_checkable
class Landmark(Protocol):
	x: float
	y: float
	z: float

LandmarkList = Sequence[Landmark]

@runtime_checkable
class HandLandmarkerResultProtocol(Protocol):
	handedness: Sequence[Sequence[Category]]
	hand_landmarks: Sequence[LandmarkList]
	hand_world_landmarks: Sequence[LandmarkList]


BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

hand_landmarker_a = HandLandmarker.create_from_options(options)


def detect_hands(frame: ImageArray, timestamp_ms: TimestampMs) -> HandLandmarkerResultProtocol:
	mp_image = mp.Image(
		image_format=mp.ImageFormat.SRGB,
		data=frame
	)
	return hand_landmarker_a.detect_for_video(mp_image, timestamp_ms)

def draw_hand_landmarks(
	image: ImageArray,
	detection_result: HandLandmarkerResultProtocol
) -> ImageArray:
	if not detection_result.hand_landmarks:
		return image

	for hand_landmarks in detection_result.hand_landmarks:
		for lm in hand_landmarks:
			h, w, _ = image.shape
			cx, cy = int(lm.x * w), int(lm.y * h)
			cv2.circle(image, (cx, cy), 4, (0, 255, 0), -1)

	return image

def extract_hand_features(
    detection_result: HandLandmarkerResultProtocol
) -> FeatureVector:
    if not detection_result.hand_landmarks:
        return np.zeros(21 * 3, dtype=np.float32)

    landmarks = detection_result.hand_landmarks[0]

    coords = np.array(
        [[lm.x, lm.y, lm.z] for lm in landmarks],
        dtype=np.float32
    )

    coords -= coords[0]  # muñeca

    return coords.flatten()

W0000 00:00:1767285359.447980  329426 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767285359.473119  329426 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
import cv2
import os

# Carpeta donde se guardarán las fotos
SAVE_DIR = "capturas"
os.makedirs(SAVE_DIR, exist_ok=True)

# Abrir cámara (0 = cámara por defecto)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("No se pudo abrir la cámara")
    exit()

print("Presiona 'Space' para tomar una foto, 'Esc' para salir.")

foto_contador = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print("No se pudo leer un frame de la cámara")
        break

    # Mostrar video en tiempo real
    cv2.imshow('Feed', frame)
    
    key = cv2.waitKey(1) & 0xFF

    if key == 27:  # Esc
        print("Saliendo...")
        break
    elif key == ' ':  # Space
        # Guardar foto
        filename = os.path.join(SAVE_DIR, f"foto_{foto_contador:03d}.png")
        cv2.imwrite(filename, frame)
        print(f"Foto guardada: {filename}")
        foto_contador += 1
#    if cv2.waitKey(10) & 0xFF == ord('q'):
#        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()


Presiona 'Space' para tomar una foto, 'Esc' para salir.
Saliendo...


In [4]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from landmark_detector.hands_landmark_detector import MediaPipeHandDetector

# # ----------------------------
# # Código original
# # ----------------------------
# BaseOptions = python.BaseOptions
# HandLandmarker = vision.HandLandmarker
# HandLandmarkerOptions = vision.HandLandmarkerOptions
# VisionRunningMode = vision.RunningMode

# options = HandLandmarkerOptions(
# 	base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
# 	running_mode=VisionRunningMode.VIDEO,
# 	num_hands=1,
# 	min_hand_detection_confidence=0.6,
# 	min_hand_presence_confidence=0.6,
# 	min_tracking_confidence=0.6
# )

# hand_landmarker = HandLandmarker.create_from_options(options)

# def detect_hands_orig(frame, timestamp_ms):
#     mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
#     return hand_landmarker.detect_for_video(mp_image, timestamp_ms)

# def draw_hand_landmarks_orig(image, detection_result):
# 	if not detection_result.hand_landmarks:
# 		return image
# 	img = image.copy()
# 	for hand_landmarks in detection_result.hand_landmarks:
# 		for lm in hand_landmarks:
# 			h, w, _ = img.shape
# 			cx, cy = int(lm.x * w), int(lm.y * h)
# 			cv2.circle(img, (cx, cy), 4, (0,255,0), -1)
# 	return img

# def extract_hand_features_orig(detection_result):
# 	if not detection_result.hand_landmarks:
# 		return np.zeros(21 * 3)
# 	hand = detection_result.hand_landmarks[0]
# 	coords = np.array([[lm.x, lm.y, lm.z] for lm in hand])
# 	coords -= coords[0]
# 	return coords.flatten()

# ----------------------------
# Código con wrapper
# ----------------------------
detector = MediaPipeHandDetector(task_path="hand_landmarker.task")

def extract_hand_features_wrapper(hands):
	if hands.num_instances == 0:
		return np.zeros(21 * 3, dtype=np.float32)
	coords = hands.select_instances([0]).relative_to(0).data
	return coords.flatten()

# ----------------------------
# Comparación
# ----------------------------
frame = cv2.imread("foto_000.png")  # ejemplo
timestamp_ms = 0

# Detectar con original
result_orig = detect_hands(frame, timestamp_ms)
features_orig = extract_hand_features(result_orig)
frame_orig = draw_hand_landmarks(frame.copy(), result_orig)

timestamp_ms += 1

hands_wrapper = detect_hands_a(frame, timestamp_ms)
features_wrapper = extract_hand_features_a(hands_wrapper)
frame_wrapper = draw_hand_landmarks_a(frame.copy(), hands_wrapper)

# Detectar con wrapper
# hands_wrapper = detector.detect(frame, timestamp_ms=timestamp_ms)
# features_wrapper = extract_hand_features_wrapper(hands_wrapper)
# frame_wrapper = hands_wrapper.draw(frame)

# ----------------------------
# Comparación numérica de features
# ----------------------------
diff = np.abs(features_orig - features_wrapper)
print("Máxima diferencia entre features:", diff.max())
print("Media diferencia entre features:", diff.mean())

# ----------------------------
# Mostrar resultados visuales
# ----------------------------
cv2.imshow("Original", frame_orig)
cv2.imshow("Wrapper", frame_wrapper)
cv2.waitKey(0)
cv2.destroyAllWindows()


W0000 00:00:1767283739.134472  322149 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767283739.154668  322149 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767283739.194333  322116 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA
Máxima diferencia entre features: 2.9802322387695312e-08
Media diferencia entre features: 1.4942907722480225e-09


# Preparing Data

## Config

In [5]:
actions = np.array([
	'jump',
	'shoot',
	'none'
])

DATA_PATH = 'HandGestureData'
frame_interval = 1 / 30  # segundos entre frames (~30 FPS)
sequence_length = 15
n_sequences_action = 10

## Creating Folders

In [17]:
for action in actions:
    for seq in range(n_sequences_action):
        os.makedirs(os.path.join(DATA_PATH, action, str(seq)), exist_ok=True)

## Collecting Data

In [18]:
import cv2
import numpy as np
import os
import time


# Abrir cámara
cap = cv2.VideoCapture(0)
cv2.namedWindow('Grabación', cv2.WINDOW_NORMAL)

print("INSTRUCCIONES: Presiona 's' para iniciar cada repetición, 'q' para salir.")

stop_recording = False  # Variable global de control

for action in actions:
    if stop_recording:
        break
    print(f"\nPróximo gesto: {action}")

    for seq in range(n_sequences_action):
        if stop_recording:
            break

        started = False

        # Esperar a que pulses 's'
        while not started:
            ret, frame = cap.read()
            if not ret:
                continue

            cv2.putText(frame, f'Próximo gesto: {action} — Presiona S para iniciar',
                        (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            cv2.imshow('Grabación', frame)

            key = cv2.waitKey(10) & 0xFF
            if key == ord('s'):
                started = True
                print(f"Iniciando grabación de {action}, secuencia {seq+1}")
            elif key == ord('q'):
                stop_recording = True
                break

        if stop_recording:
            break

        # Grabación por intervalos de tiempo
        frames_captured = 0
        last_time = time.time()
        while frames_captured < sequence_length:
            if stop_recording:
                break

            current_time = time.time()
            if current_time - last_time >= frame_interval:
                last_time = current_time

                ret, frame = cap.read()
                if not ret:
                    continue

                timestamp = int(current_time * 1000)
                result = detect_hands(frame, timestamp)
                frame_display = draw_hand_landmarks(frame, result)

                cv2.putText(frame_display, f'{action} — secuencia {seq+1} frame {frames_captured+1}/{sequence_length}',
                            (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
                cv2.imshow('Grabación', frame_display)

                features = extract_hand_features(result)
                np.save(os.path.join(DATA_PATH, action, str(seq), f'{frames_captured}.npy'), features)

                frames_captured += 1

            # Revisar tecla q cada frame
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                stop_recording = True
                break

        if stop_recording:
            break

        print(f"Secuencia completada: {seq+1}/{n_sequences_action}")
        print("Pulsa 's' para iniciar la siguiente repetición...")

cap.release()
cv2.destroyAllWindows()
print("Grabación terminada")


INSTRUCCIONES: Presiona 's' para iniciar cada repetición, 'q' para salir.

Próximo gesto: jump
Iniciando grabación de jump, secuencia 1
Secuencia completada: 1/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 2
Secuencia completada: 2/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 3
Secuencia completada: 3/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 4
Secuencia completada: 4/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 5
Secuencia completada: 5/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 6
Secuencia completada: 6/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 7
Secuencia completada: 7/10
Pulsa 's' para iniciar la siguiente repetición...
Iniciando grabación de jump, secuencia 8
Secuencia completada: 8/10
Pulsa 's' p

## Data Augmentation

In [19]:
import os
import numpy as np
from typing import Dict, Tuple, List

# --- Transformaciones individuales ---
def add_spatial_noise(seq: np.ndarray, std: float) -> np.ndarray:
    return seq + np.random.normal(0, std, seq.shape)

def scale_sequence(seq: np.ndarray, scale: float) -> np.ndarray:
    return seq * scale

def frame_dropout(seq: np.ndarray, idx: int) -> np.ndarray:
    seq = np.array(seq)
    seq[idx] = 0
    return seq

from scipy.interpolate import interp1d

def temporal_interpolation(seq: np.ndarray, sub_start: int, sub_end: int, sequence_length: int) -> np.ndarray:
    sub_seq = seq[sub_start:sub_end]
    x_old = np.linspace(0, 1, num=len(sub_seq))
    x_new = np.linspace(0, 1, num=sequence_length)
    f = interp1d(x_old, sub_seq, axis=0)
    return f(x_new)

def temporal_padding(seq: np.ndarray, sub_start: int, sub_end: int, sequence_length: int) -> np.ndarray:
    sub_seq = seq[sub_start:sub_end]
    pad_len = sequence_length - len(sub_seq)
    pad_front = pad_len // 2
    pad_back = pad_len - pad_front
    return np.vstack([
        np.tile(sub_seq[0], (pad_front, 1)),
        sub_seq,
        np.tile(sub_seq[-1], (pad_back, 1))
    ])

# --- Función principal de augmentación ---
def augment_sequence(seq: np.ndarray,
                     sequence_length: int,
                     noise_std: float = 0.01,
                     scale_range: Tuple[float, float] = (0.9, 1.1),
                     dropout_prob: float = 0.3,
                     temporal_prob: float = 0.5) -> np.ndarray:
    seq = np.array(seq)
    orig_len = len(seq)

    # Ruido espacial
    if np.random.rand() < 1.0:
        seq = add_spatial_noise(seq, noise_std)

    # Escalado
    if np.random.rand() < 1.0:
        scale = np.random.uniform(*scale_range)
        seq = scale_sequence(seq, scale)

    # Frame dropout
    if np.random.rand() < dropout_prob:
        idx = np.random.randint(0, orig_len)
        seq = frame_dropout(seq, idx)

    # Transformaciones temporales
    if np.random.rand() < temporal_prob:
        sub_start = np.random.randint(0, orig_len // 2)
        sub_end = sub_start + np.random.randint(orig_len // 2, orig_len)
        sub_end = min(sub_end, orig_len)

        if np.random.rand() < 0.5:
            seq = temporal_interpolation(seq, sub_start, sub_end, sequence_length)
        else:
            seq = temporal_padding(seq, sub_start, sub_end, sequence_length)

    return seq.astype(np.float32)  # asegurar consistencia de tipo

# Creating Dataset

In [20]:
augmentation_counts: Dict[str, int] = {
    'jump': 30,
    'shoot': 30,
    'none': 60
}

label_map: Dict[str, int] = {label: i for i, label in enumerate(actions)}

sequences: List[np.ndarray] = []
labels: List[int] = []

for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    for seq_folder in os.listdir(action_path):
        window = [np.load(os.path.join(action_path, seq_folder, f'{frame}.npy'))
                  for frame in range(sequence_length)]
        window = np.array(window, dtype=np.float32)

        # Secuencia original
        sequences.append(window)
        labels.append(label_map[action])

        # Secuencias aumentadas
        n_augments = augmentation_counts.get(action, 1)
        for _ in range(n_augments):
            sequences.append(augment_sequence(window, sequence_length, temporal_prob=0))
            labels.append(label_map[action])

# Guardar dataset
np.savez_compressed(
    'hand_gesture_dataset.npz',
    X=np.stack(sequences),  # usar stack asegura forma uniforme
    y=np.array(labels, dtype=np.int32)
)
print("Dataset guardado en 'hand_gesture_dataset.npz' con X.shape =", np.stack(sequences).shape)

Dataset guardado en 'hand_gesture_dataset.npz' con X.shape = (1230, 15, 63)


# Creating Splits

In [21]:
# 3️⃣ Cargar dataset desde archivo
data = np.load('hand_gesture_dataset.npz')
X = data['X']
y_labels = data['y']

# Convertir a one-hot
y = to_categorical(y_labels, num_classes=len(actions))

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y_labels
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (1045, 15, 63) X_test: (185, 15, 63)


# Creating Model

In [22]:
MODEL_EXPORT_NAME = 'hand_gesture_model.h5'

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed

model = Sequential([
    TimeDistributed(Dense(64, activation='relu'),
                    input_shape=(sequence_length, X.shape[2])),
    Dropout(0.3),

    GRU(64),
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_1              │ (None, 15, 64)         │         4,096 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,235 (122.01 KB)

 Trainable params: 31,235 (122.01 KB)

 Non-trainable params: 0 (0.00 B)

# Training

In [24]:
model.fit(
    X_train, y_train,
    epochs=100,
    validation_split=0.2,
    batch_size=16
)

model.save(MODEL_EXPORT_NAME)

Epoch 1/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.5766 - loss: 0.8783 - val_accuracy: 0.6364 - val_loss: 0.6559
Epoch 2/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8002 - loss: 0.4957 - val_accuracy: 0.9856 - val_loss: 0.2050
Epoch 3/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9689 - loss: 0.1204 - val_accuracy: 1.0000 - val_loss: 0.0133
Epoch 4/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9988 - loss: 0.0145 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 5/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 9.7936e-04
Epoch 6/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9952 - loss: 0.0212 - val_accuracy: 1.0000 - val_loss: 0.0030
Epoch 7/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9761 - loss: 0.1030 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 8/100
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 0.0035 - val_accuracy:

# Metrics

In [25]:

from tensorflow.keras.models import load_model

model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step
Accuracy: 1.0
[[47  0  0]
 [ 0 46  0]
 [ 0  0 92]]


# Testing

In [26]:

from tensorflow.keras.models import load_model

model = load_model(MODEL_EXPORT_NAME)

import time

sequence = []
pred_buffer = []
threshold = 0.85

cap = cv2.VideoCapture(0)

last_timestamp = int(time.time() * 1000)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # timestamp monotónico en ms
    timestamp_ms = int(time.time() * 1000)
    if timestamp_ms <= last_timestamp:
        timestamp_ms = last_timestamp + 1
    last_timestamp = timestamp_ms

    result = detect_hands(frame, timestamp_ms)
    frame = draw_hand_landmarks(frame, result)

    features = extract_hand_features(result)
    sequence.append(features)
    sequence = sequence[-sequence_length:]

    if len(sequence) == sequence_length:
        res = model.predict(np.expand_dims(sequence, axis=0))[0]
        pred_buffer.append(res)

        avg = np.mean(pred_buffer[-5:], axis=0)
        idx = np.argmax(avg)

        if avg[idx] > threshold and actions[idx] != 'none':
            cv2.putText(frame, actions[idx],
                        (50,50), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0,255,0), 3)
            # 🔥 aquí llamas a tu lógica del juego

    cv2.imshow('Feed', frame)
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━